# Create Subset of 9,128 Slides from train_images

This notebook reads `train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv` (9,128 slides after noisy label removal) and copies only those slides from the full `train_images` folder into a new subset folder.

In [ ]:
# Setup and Mount Google Drive
import os
import shutil
import pandas as pd
from tqdm import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Google Drive mounted!')
except:
    IN_COLAB = False
    print('Not in Colab environment')

Mounted at /content/drive
Google Drive mounted!


In [ ]:
# Configuration - UPDATE THESE PATHS
CONFIG = {
    'csv_path': '/content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/Kaggle-PANDA-1st-place-solution-master/input/train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv',
    'src_folder': '/content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/train_images',
    'dst_folder': '/content/drive/MyDrive/PANDA_Subset_9128/images',
    'image_ext': '.tiff',
}

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

Configuration:
  csv_path: /content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/Kaggle-PANDA-1st-place-solution-master/input/train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv
  src_folder: /content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/train_images
  dst_folder: /content/drive/MyDrive/PANDA_Subset_9128/images
  image_ext: .tiff


In [ ]:
# Step 1: Load CSV and extract slide IDs
print('='*50)
print('STEP 1: LOADING CSV')
print('='*50)

df = pd.read_csv(CONFIG['csv_path'])
slide_ids = df['image_id'].unique().tolist()

print(f'Total slides in CSV: {len(slide_ids)}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nSample IDs:')
for sid in slide_ids[:5]:
    print(f'  {sid}')

STEP 1: LOADING CSV
Total slides in CSV: 9128

Columns: ['Unnamed: 0', 'image_id', 'data_provider', 'isup_grade', 'gleason_score', 'kfold', 'probs_raw', 'probs_without_tta_raw', 'preds', 'preds_without_tta']

Sample IDs:
  de4966a35d76fc988a17d78d895b2130
  5a2deec84849fdc2216bab61452451b7
  baf9bcfaaaf14467f766dcad728948da
  bb6ea8e115a5719d2644a63919858ea9
  10b00bbaff7224b83537ccb02e7e1416


In [ ]:
# Step 2: Show grade distribution
print('='*50)
print('STEP 2: GRADE DISTRIBUTION')
print('='*50)

print('\nISUP Grade distribution:')
print(df['isup_grade'].value_counts().sort_index())

print('\nData provider distribution:')
print(df['data_provider'].value_counts())

print('\nGleason score distribution:')
print(df['gleason_score'].value_counts().sort_index())

STEP 2: GRADE DISTRIBUTION

ISUP Grade distribution:
isup_grade
0    2758
1    2438
2    1133
3     825
4     984
5     990
Name: count, dtype: int64

Data provider distribution:
data_provider
karolinska    5121
radboud       4007
Name: count, dtype: int64

Gleason score distribution:
gleason_score
0+0         1904
3+3         2438
3+4         1133
3+5           30
4+3          825
4+4          927
4+5          685
5+3           27
5+4          209
5+5           96
negative     854
Name: count, dtype: int64


In [ ]:
# Step 3: Check source folder
print('='*50)
print('STEP 3: CHECKING SOURCE FOLDER')
print('='*50)

src_folder = CONFIG['src_folder']
if os.path.exists(src_folder):
    src_files = os.listdir(src_folder)
    print(f'Source folder exists: {src_folder}')
    print(f'Total files in source: {len(src_files)}')
    print(f'Sample files: {src_files[:5]}')
else:
    print(f'ERROR: Source folder not found: {src_folder}')
    print('Please update CONFIG["src_folder"] with the correct path.')

STEP 3: CHECKING SOURCE FOLDER
Source folder exists: /content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/train_images
Total files in source: 10616
Sample files: ['e920b4b7fadf9463723ecd4e5755c485.tiff', 'e9193d66b4eb4c6778efdeb776c5caa3.tiff', 'e9277e645833b34f708f1e0a991cca01.tiff', 'e92dfc70e301a7e97540ac17cb56cdae.tiff', 'e93861af06ca8a35bc36d1013c1c269a.tiff']


In [ ]:
# Step 4: Copy slides to new subset folder
print('='*50)
print('STEP 4: COPYING SLIDES')
print('='*50)

dst_folder = CONFIG['dst_folder']
os.makedirs(dst_folder, exist_ok=True)
print(f'Destination folder: {dst_folder}')

copied = 0
skipped = 0
missing = []

for img_id in tqdm(slide_ids, desc='Copying slides'):
    filename = img_id + CONFIG['image_ext']
    src_path = os.path.join(src_folder, filename)
    dst_path = os.path.join(dst_folder, filename)

    # Skip if already copied
    if os.path.exists(dst_path):
        skipped += 1
        continue

    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        copied += 1
    else:
        missing.append(img_id)

print(f'\nCopied: {copied}')
print(f'Skipped (already exist): {skipped}')

if missing:
    print(f'\nMissing {len(missing)} slides (not found in source):')
    for m in missing[:10]:
        print(f'  - {m}')
    if len(missing) > 10:
        print(f'  ... and {len(missing) - 10} more')
else:
    print('\nAll slides found and copied!')

STEP 4: COPYING SLIDES
Destination folder: /content/drive/MyDrive/PANDA_Subset_9128/images


Copying slides: 100%|██████████| 9128/9128 [3:27:43<00:00,  1.37s/it]


Copied: 9128
Skipped (already exist): 0

All slides found and copied!


In [ ]:
# Step 5: Verify the subset
print('='*50)
print('STEP 5: VERIFICATION')
print('='*50)

dst_files = os.listdir(dst_folder)
dst_ids = [os.path.splitext(f)[0] for f in dst_files if not f.startswith('.')]

print(f'Files in subset folder: {len(dst_files)}')
print(f'Expected: {len(slide_ids)}')

if len(dst_files) == len(slide_ids):
    print('\nAll slides copied successfully!')
else:
    print(f'\nDifference: {len(slide_ids) - len(dst_files)} slides missing')

# Calculate total size
total_size = sum(os.path.getsize(os.path.join(dst_folder, f)) for f in dst_files)
print(f'\nTotal subset size: {total_size / (1024**3):.2f} GB')

STEP 5: VERIFICATION
Files in subset folder: 9128
Expected: 9128

All slides copied successfully!

Total subset size: 311.63 GB


In [ ]:
# Summary
print('='*50)
print('SUMMARY')
print('='*50)

print(f'''
Subset Created Successfully!

Source: {CONFIG['src_folder']}
Destination: {CONFIG['dst_folder']}
CSV: {CONFIG['csv_path']}

Total slides copied: {len(dst_files)}
Total size: {total_size / (1024**3):.2f} GB

Grade Distribution:
{df['isup_grade'].value_counts().sort_index().to_string()}

Data Provider Distribution:
{df['data_provider'].value_counts().to_string()}
''')

SUMMARY

Subset Created Successfully!

Source: /content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/train_images
Destination: /content/drive/MyDrive/PANDA_Subset_9128/images
CSV: /content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/Kaggle-PANDA-1st-place-solution-master/input/train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv

Total slides copied: 9128
Total size: 311.63 GB

Grade Distribution:
isup_grade
0    2758
1    2438
2    1133
3     825
4     984
5     990

Data Provider Distribution:
data_provider
karolinska    5121
radboud       4007

